# Filter on toxic interaction

In [11]:
import pandas as pd

def load_toxicity_keywords(mapping_file: str, toxicity_index: int = 10) -> list:
    """Extract and clean toxicity type keywords mapped to a given index."""
    df = pd.read_excel(mapping_file)
    toxicity_raw = df[df['merged DDI type index'] == toxicity_index]["Origin DDI's type"]

    # Clean: remove single quotes and commas, strip whitespace, and lowercase
    toxicity_keywords = (
        toxicity_raw
        .astype(str)
        .str.replace("'", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .str.lower()
        .tolist()
    )

    return toxicity_keywords


def match_toxicity_types(interaction: str, toxicity_keywords: list) -> list:
    """Return matched toxicity keywords in the interaction string."""
    interaction_lower = interaction.lower()
    return [kw for kw in toxicity_keywords if kw in interaction_lower]

def filter_and_label_toxicity(ddi_file: str, toxicity_keywords: list) -> pd.DataFrame:
    """Filter rows with any toxicity keyword and label matched ones."""
    ddi_df = pd.read_csv(ddi_file)
    ddi_df['interaction_type'] = ddi_df['interaction_type'].astype(str)

    # Apply matching
    ddi_df['matched_toxicity_types'] = ddi_df['interaction_type'].apply(
        lambda x: match_toxicity_types(x, toxicity_keywords)
    )

    # Filter rows with at least one match
    filtered_df = ddi_df[ddi_df['matched_toxicity_types'].apply(lambda x: len(x) > 0)].copy()

    # Join matched types into a string for saving
    filtered_df['matched_toxicity_types'] = filtered_df['matched_toxicity_types'].apply(lambda x: ';'.join(x))

    return filtered_df

In [ ]:
ddi_file = 'data/DDI_data.csv'
mapping_file = 'data/DDI_types.xlsx'
output_file = 'data/toxicity_ddi_data.csv'

toxicity_keywords = load_toxicity_keywords(mapping_file)
print(f"Loaded {len(toxicity_keywords)} toxicity keywords.")
print(toxicity_keywords)

toxicity_df = filter_and_label_toxicity(ddi_file, toxicity_keywords)

toxicity_df.to_csv(output_file, index=False)
print(f"Saved filtered toxicity interactions to {output_file}")

Loaded 10 toxicity keywords.
['cardiotoxic activities', 'nephrotoxic activities', 'risk or severity of renal failure', 'hepatotoxic activities', 'neurotoxic activities', 'hypotensive nephrotoxic and hyperkalemic activities', 'central neurotoxic activities', 'risk or severity of ototoxicity and nephrotoxicity', 'ototoxic activities', 'risk or severity of pulmonary toxicity']
Saved filtered toxicity interactions to data/toxicity_ddi_data.csv


In [15]:
toxicity_df

,drug1_id,drug2_id,drug1_name,drug2_name,interaction_type,matched_toxicity_types
298,DB00014,DB00390,Goserelin,Digoxin,cardiotoxic activities,cardiotoxic activities
299,DB00014,DB00511,Goserelin,Acetyldigitoxin,cardiotoxic activities,cardiotoxic activities
300,DB00014,DB01078,Goserelin,Deslanoside,cardiotoxic activities,cardiotoxic activities
301,DB00014,DB01092,Goserelin,Ouabain,cardiotoxic activities,cardiotoxic activities
302,DB00014,DB01396,Goserelin,Digitoxin,cardiotoxic activities,cardiotoxic activities
...,...,...,...,...,...,...
221900,DB09449,DB13167,Sodium phosphate,Alclofenac,nephrotoxic activities,nephrotoxic activities
221901,DB09449,DB13346,Sodium phosphate,Bufexamac,nephrotoxic activities,nephrotoxic activities
221902,DB09449,DB13783,Sodium phosphate,Acemetacin,nephrotoxic activities,nephrotoxic activities
222542,DB12615,DB00447,Plazomicin,Loracarbef,nephrotoxic activities,nephrotoxic activities


## TWOSIDES

In [16]:
import pandas as pd
from tqdm import tqdm

# Path to full TWOSIDES and reduced output file
input_path = "/Users/chiphan/Documents/Chi/1-Learning/2025-Spring/COMP4010-DataViz/Assignment/Project/Project-2/toxic-ddi-visualization/data/TWOSIDES.csv"
output_path = "/Users/chiphan/Documents/Chi/1-Learning/2025-Spring/COMP4010-DataViz/Assignment/Project/Project-2/toxic-ddi-visualization/data/twosides_minimal.csv"

# === Columns to keep ===
columns_to_keep = [
    "drug_1_concept_name",
    "drug_2_concept_name",
    "condition_concept_name",
    "PRR"
]

# === Count total lines for progress bar (optional but helps)
with open(input_path, "r", encoding="utf-8") as f:
    total_lines = sum(1 for _ in f)

# === Read in chunks with progress
chunksize = 100_000
reader = pd.read_csv(input_path, usecols=columns_to_keep, chunksize=chunksize, low_memory=False)

filtered_chunks = []
for chunk in tqdm(reader, total=total_lines // chunksize + 1, desc="Creating minimal TWOSIDES"):
    filtered_chunks.append(chunk)

# === Combine & save
pd.concat(filtered_chunks).to_csv(output_path, index=False)
print(f"✅ Saved to: {output_path}")

Creating minimal TWOSIDES: 100%|██████████| 430/430 [00:16<00:00, 25.40it/s]


✅ Saved to: /Users/chiphan/Documents/Chi/1-Learning/2025-Spring/COMP4010-DataViz/Assignment/Project/Project-2/toxic-ddi-visualization/data/twosides_minimal.csv


In [17]:
twosides_path = "/Users/chiphan/Documents/Chi/1-Learning/2025-Spring/COMP4010-DataViz/Assignment/Project/Project-2/toxic-ddi-visualization/data/twosides_minimal.csv"
twosides_df = pd.read_csv(twosides_path)
twosides_df.head()

/var/folders/48/t9f5yscs7qd6km5jw6dzxprw0000gn/T/ipykernel_12212/1560522319.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  twosides_df = pd.read_csv(twosides_path)


,drug_1_concept_name,drug_2_concept_name,condition_concept_name,PRR
0,Temazepam,sildenafil,Arthralgia,2.91667
1,Bumetanide,Oxytocin,Arthralgia,5.0
2,POLYETHYLENE GLYCOL 3350,Hydroxychloroquine,Arthralgia,3.0
3,Tamoxifen,Prednisone,Diarrhoea,5.14286
4,Temazepam,sildenafil,Diarrhoea,0.540541


In [18]:
# Load twosides minimal file
twosides = pd.read_csv(twosides_path)

# Filter condition_concept_name that contains "toxicity"
tox_filtered = twosides[twosides["condition_concept_name"].str.contains("toxicity", case=False, na=False)]

# Save or return
tox_filtered.to_csv("/Users/chiphan/Documents/Chi/1-Learning/2025-Spring/COMP4010-DataViz/Assignment/Project/Project-2/toxic-ddi-visualization/data/twosides_toxicity_only.csv", index=False)
print("✅ Saved filtered TWOSIDES with toxicity-related conditions to twosides_toxicity_only.csv")


/var/folders/48/t9f5yscs7qd6km5jw6dzxprw0000gn/T/ipykernel_12212/2505841845.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  twosides = pd.read_csv(twosides_path)


✅ Saved filtered TWOSIDES with toxicity-related conditions to twosides_toxicity_only.csv


In [21]:
# unique values in condition_concept_name
unique_conditions = tox_filtered["condition_concept_name"].unique()
print(f"Total unique conditions: {len(unique_conditions)}")
print(unique_conditions)

Total unique conditions: 21
['Toxicity to various agents' 'Drug toxicity' 'Therapeutic agent toxicity'
 'Hepatotoxicity' 'Neurotoxicity' 'Cardiotoxicity' 'Skin toxicity'
 'Haematotoxicity' 'Pulmonary toxicity' 'Ototoxicity'
 'Gastrointestinal toxicity' 'Bone marrow toxicity'
 'Mitochondrial toxicity' 'Ocular toxicity' 'Retinal toxicity'
 'Anticonvulsant toxicity' 'Nail toxicity' 'Platelet toxicity'
 'Mucosal toxicity' 'Chemotherapy neurotoxicity attenuation'
 'Neuromuscular toxicity']


# DDI

In [ ]:
# Load your DDI file
ddi_path = "/Users/chiphan/Documents/Chi/1-Learning/2025-Spring/COMP4010-DataViz/Assignment/Project/Project-2/toxic-ddi-visualization/data/toxicity_ddi_data.csv"

ddi = pd.read_csv(ddi_path)

# Define mapping rules
toxicity_mapping = {
    "cardiotoxic": "cardiotoxicity",
    "nephrotoxic": "nephrotoxicity",
    "renal failure": "nephrotoxicity",
    "hepatotoxic": "hepatotoxicity",
    "neurotoxic": "neurotoxicity",
    "central neurotoxic": "neurotoxicity",
    "ototoxic": "ototoxicity",
    "pulmonary toxicity": "pulmonary toxicity"
}

# Normalize function
def normalize_interaction_type(text):
    text = str(text).lower()
    for key, value in toxicity_mapping.items():
        if key in text:
            return value
    return text  # fallback to original

# Apply normalization
ddi["interaction_type_normalized"] = ddi["interaction_type"].apply(normalize_interaction_type)

# Save or inspect
ddi.to_csv("/Users/chiphan/Documents/Chi/1-Learning/2025-Spring/COMP4010-DataViz/Assignment/Project/Project-2/toxic-ddi-visualization/data/ddi_normalized.csv", index=False)
print(" Saved normalized DDI interaction types to ddi_normalized.csv")


 Saved normalized DDI interaction types to ddi_normalized.csv


In [25]:
import pandas as pd

# === Load datasets ===
ddi = pd.read_csv("/Users/chiphan/Documents/Chi/1-Learning/2025-Spring/COMP4010-DataViz/Assignment/Project/Project-2/toxic-ddi-visualization/data/toxicity_ddi_normalized.csv")
twosides = pd.read_csv("/Users/chiphan/Documents/Chi/1-Learning/2025-Spring/COMP4010-DataViz/Assignment/Project/Project-2/toxic-ddi-visualization/data/twosides_toxicity_only.csv")

# === Normalize all names for matching ===
def clean(text):
    return str(text).strip().lower()

ddi["drug1_clean"] = ddi["drug1_name"].apply(clean)
ddi["drug2_clean"] = ddi["drug2_name"].apply(clean)
ddi["tox_clean"] = ddi["interaction_type_normalized"].apply(clean)

twosides["drug_1_clean"] = twosides["drug_1_concept_name"].apply(clean)
twosides["drug_2_clean"] = twosides["drug_2_concept_name"].apply(clean)
twosides["condition_clean"] = twosides["condition_concept_name"].apply(clean)

# === Build a set of lookups: {(drugA, drugB, toxicity): PRR} ===
from collections import defaultdict

prr_lookup = defaultdict(list)

for _, row in twosides.iterrows():
    drugA = row["drug_1_clean"]
    drugB = row["drug_2_clean"]
    condition = row["condition_clean"]
    key = (frozenset([drugA, drugB]), condition)
    prr_lookup[key].append(row["PRR"])

# === Match and extract max PRR per row ===
def find_prr(row):
    pair = frozenset([row["drug1_clean"], row["drug2_clean"]])
    tox = row["tox_clean"]
    
    matched = [max(prrs) for (drugs, cond), prrs in prr_lookup.items()
               if drugs == pair and tox in cond]  # substring match

    return matched[0] if matched else None

ddi["matched_prr"] = ddi.apply(find_prr, axis=1)

# === Save final result ===
ddi.drop(columns=["drug1_clean", "drug2_clean", "tox_clean"], inplace=True)
ddi.to_csv("/Users/chiphan/Documents/Chi/1-Learning/2025-Spring/COMP4010-DataViz/Assignment/Project/Project-2/toxic-ddi-visualization/data/toxicity_ddi_with_matched_prr.csv", index=False)
print("✅ Saved to toxicity_ddi_with_matched_prr.csv")


✅ Saved to toxicity_ddi_with_matched_prr.csv
